# SEVIRI SSL: observations and final clip dataset

The current training selection contains **17,568 clips** from 93 observation dates in January–June 2022. This notebook presents archive coverage, the selected crop distribution and the sampling procedure. The complete reproducible specification is in [SEVIRI_SSL_DATASET.md](../SEVIRI_SSL_DATASET.md).

Run the cells in order with **Python (deep_convection)**. Training results are presented in [study_seviri_ssl_training.ipynb](study_seviri_ssl_training.ipynb).

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib, json, sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/emma_gpm").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from emma_gpm import ssl_archive_coverage, ssl_dashboard
archive_views = importlib.reload(ssl_archive_coverage)
views = importlib.reload(ssl_dashboard)

CURRENT_DATASET = Path("/data1/deep_convection_data/ssl_continuous_iou020_20261111")
DATASET_RESULTS = ROOT / "results/ssl_continuous_iou020_20261111"
OUTPUT = ROOT / "results/dataset_study_final_20261111"
OUTPUT.mkdir(exist_ok=True)
training = pd.read_csv(CURRENT_DATASET / "training/dataset/train.csv")
validation = pd.read_csv(CURRENT_DATASET / "extended/dataset/validation.csv")
protocol = json.loads((CURRENT_DATASET / "training/dataset/protocol.json").read_text())
stats = json.loads((CURRENT_DATASET / "training/dataset/channel_stats.json").read_text())

def figure(fig, filename):
    fig.savefig(OUTPUT / filename, dpi=140, bbox_inches="tight")
    display(fig)
    plt.close(fig)


## Archive coverage

In [ ]:
ARCHIVE = Path("/data1/deep_convection_data/MSG-SEVIRI")
INVENTORY_FILE = OUTPUT / "archive_inventory_current.json"
REFRESH_ARCHIVE_INVENTORY = False
if REFRESH_ARCHIVE_INVENTORY or not INVENTORY_FILE.exists():
    from emma_gpm.ssl_dataset import inventory_upload
    archive_inventory = inventory_upload(ARCHIVE, stable_seconds=60)
    archive_inventory.update(root=str(ARCHIVE), captured_at=datetime.now(timezone.utc).isoformat())
    INVENTORY_FILE.write_text(json.dumps(archive_inventory))
else:
    archive_inventory = json.loads(INVENTORY_FILE.read_text())
archive_daily, archive_totals = archive_views.archive_coverage(archive_inventory)
archive_daily.to_csv(OUTPUT / "archive_daily_coverage.csv", index=False)
(OUTPUT / "archive_coverage_summary.json").write_text(json.dumps(archive_totals, indent=2) + "\n")
figure(archive_views.archive_coverage_figure(
    archive_daily, archive_totals,
    selected={"Training observations": training, "Validation observations": validation},
    training_samples=len(training)), "archive_temporal_coverage.png")
display(Markdown(
    "**Inventory recorded:** " + pd.Timestamp(archive_inventory["captured_at"]).tz_convert("Europe/Paris").strftime("%d %B %Y, %H:%M %Z") +
    f" · **Observation dates:** {archive_totals['observation_dates']:,}."
))


## Final training selection: 17,568 clips

In [ ]:
coverage, origins = views.manifest_coverage_figure(
    training, CURRENT_DATASET / "sources", title=f"Training selection · {len(training):,} clips"
)
figure(coverage, "training_crop_coverage.png")
origins.to_csv(OUTPUT / "training_crop_origins.csv", index=False)
video = DATASET_RESULTS / "sampling_20220506_20220512_28s.mp4"
display(Image(filename=str(video.with_suffix(".png")), width=1100))
display(Markdown(
    "[**Open the 28-second weekly animation**](../results/ssl_continuous_iou020_20261111/sampling_20220506_20220512_28s.mp4)"
))


## Dataset description

### Observations, geographic domain and clip contents

Downloaded SEVIRI observations are stored in daily Zarr archives on the native geostationary grid. A **frame** is one observation time containing the seven thermal channels. An inventory records timestamps, coordinates and physically present data blocks. The current training manifest uses 93 dates between **1 January and 29 June 2022**, with **2,928 clips from each month**.

The geographic domain is **20°W–40°E, 30–70°N**. Every pixel centre in an accepted crop lies inside these bounds. A **clip** is identified by its first timestamp and spatial origin and contains **16 consecutive frames at five-minute cadence**, seven channels and **224 × 224 pixels**. Its first and last observations are separated by 75 minutes. All required pixels are finite and within 100–400 K. Coordinate axes determine the north-up, west-left orientation.

The channel order is **WV_062, WV_073, IR_087, IR_097, IR_108, IR_120, IR_134**, corresponding to nominal wavelengths of 6.2, 7.3, 8.7, 9.7, 10.8, 12.0 and 13.4 μm. **IR108** denotes IR_108 in the diagnostic figures.

### Continuous proposals, overlap constraints and stopping rule

A proposal chooses a complete five-minute start uniformly among eligible starts, then chooses a spatial origin uniformly among the valid integer-pixel origins for that window. Windows can cross midnight and year boundaries when all observations remain available and eligible for training. Missing observations and reserved dates interrupt the eligible sequence.

Spatial **intersection over union (IoU)** measures the common crop area relative to the area covered by either crop. Every pair of selected clips sharing an observed frame satisfies **IoU ≤ 0.20**. A new clip must also share at most **20% of its pixel–timestamp positions** with each existing clip. This second criterion combines common area with the number of shared frames. The fixed input clips retain their established pairwise constraints. Both preceding and following clips enter the overlap checks.

Each proposal block contains **1,024 individually evaluated candidates**. A day stops after **ten consecutive blocks with acceptance below 10%**, with a ceiling of **32,768 proposals per day**. A block reaching 10% acceptance resets the consecutive-block counter. The final selection preserves its fixed input clips and balances the monthly totals, using the available dates and six-hour bins within each month. The realized training selection contains **17,568 clips**, including **134 spanning midnight**, and uses **26,735 distinct SEVIRI observation times**.

### Validation, normalization and materialized files

The primary validation contains **512 clips**, all included in an extended validation of **10,240 clips from 20 dates**. Training observations remain at least 24 hours from the reserved validation dates; complete clip intervals are checked against these reservations. The two validation manifests are fixed for comparisons between models.

Training and validation clips are saved as complete, losslessly compressed float16 Kelvin arrays. Training reads these materialized files. A single global mean and standard deviation are calculated over all selected training clips, all 16 frames, all seven channels and all pixels: **256.6 K** and **19.5 K**, respectively. These training constants also normalize validation, using the full precision stored in the statistics file.

### Archive counts and figure conventions

The archive overview includes every inventoried observation time with seven-channel data blocks supporting at least one crop inside the geographic domain. **Coverage at five-minute cadence** sums one five-minute interval per distinct observation, including disconnected periods. Monthly coverage is the fraction of calendar five-minute slots represented by these observations.

**Geometric clip candidates** counts every complete 16-frame window at a one-frame temporal stride, combined with every admissible 224 × 224 crop origin at a one-pixel stride in both spatial directions. Compatible adjacent daily archives contribute windows across midnight. Nearby candidates share many pixels and frames. This count uses geographic bounds and physical block availability, before pixelwise radiometric screening, split reservations and pairwise overlap limits. **Selected training clips** is the size of the current manifest on its 93 training dates. The two counts therefore describe the archive's geometric combinations and the actual selected collection.

In the spatial panel, each small orange point locates a distinct crop's north-west origin after orientation. Orange outlines trace actual crop boundaries at representative origins spaced approximately 5° along all four edges of the origin distribution. The histograms count selected clips by start date and UTC start hour. The animation still shows the same manifest on the native geostationary grid; its linked video covers 6–12 May 2022 with fifteen-minute display intervals.

## Temporary reference: historical variability

This reference supports comparison of the initial SSL training experiments. Their **historical variability score** averages the absolute IR108 brightness-temperature differences over the nine consecutive frame pairs in the ten-frame historical input and over all crop pixels. Its unit is kelvin per observed five-minute difference. Both cloud displacement and radiometric changes contribute to this score.

The earlier variability-stratified sampling combines a 50% uniform component with a 50% component weighted across low, medium and high variability using relative weights of 1:2:4. When interpreting those training results, retain the corresponding sampling distribution and validation population. Each evaluated checkpoint uses the global normalization constants from its own training dataset.